# Ch 29 · Gradients, Sampling-Based Optimization and Parallel System Identification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/29-gradients-and-system-identification.ipynb)

Companion notebook for the chapter page `chapters/29-gradients-and-system-identification.html` ([page](../chapters/29-gradients-and-system-identification.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

This notebook holds every example of Chapter 29 in page order. Each cell is the same file as `examples/29-gradients-and-system-identification/exN_*.py`. On a GPU runtime the scripts pick the big budgets (NWORLD 1024); on CPU they use small ones. No example needs a helper module.

Examples 29.2-29.4 use the private module `mujoco_warp._src.diff` (it may change between versions).

## 1-2. Reverse mode and `wp.Tape` on a plain kernel
Forward under the tape, seed `y.grad = 1`, `tape.backward()`, read `x.grad`. Note that the backward *consumes* the output cotangent of an overwritten array and *accumulates* into inputs.

In [ ]:
# Example 29.1 - ex1_tape_basics.py
"""Example 29.1 - reverse-mode autodiff with wp.Tape on a two-line kernel, y = x**2.

The forward kernel runs under a tape. Warp generated an adjoint (backward) kernel for it when it
compiled the module. We seed the output cotangent y.grad = 1, call tape.backward(), and read
x.grad = dy/dx = 2x. A central finite difference checks the answer.
Run: python ex1_tape_basics.py
"""
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

np.set_printoptions(precision=4, suppress=True)


@wp.kernel
def square(x: wp.array[float], y: wp.array[float]):
    i = wp.tid()
    y[i] = x[i] * x[i]


x_np = np.array([-1.0, 0.5, 1.5, 3.0], dtype=np.float32)
x = wp.array(x_np, dtype=float, requires_grad=True)   # leaf: we want dL/dx
y = wp.zeros(4, dtype=float, requires_grad=True)

tape = wp.Tape()
with tape:                                             # 1. forward, recorded
    wp.launch(square, dim=4, inputs=[x], outputs=[y])
print("recorded launches:", len(tape.launches))
print("y      =", y.numpy())

y.grad.fill_(1.0)                                      # 2. seed: L = sum(y), dL/dy = 1
tape.backward()                                        # 3. run the adjoint kernels in reverse
print("x.grad =", x.grad.numpy(), " (expected 2x)")

# 4. central finite differences, one input at a time, float64 on the host
eps = 1e-3
f = lambda v: v * v
fd = (f(x_np.astype(np.float64) + eps) - f(x_np.astype(np.float64) - eps)) / (2 * eps)
print("FD     =", fd)
print("max |tape - FD| =", float(np.abs(x.grad.numpy() - fd).max()))

# 5. the backward CONSUMED the output cotangent: the forward overwrote y, so y.grad is now the
#    cotangent of y's old contents (0). Re-seed it and x.grad ACCUMULATES (+=), it is not reset.
print("y.grad after backward =", y.grad.numpy())
y.grad.fill_(1.0)
tape.backward()
print("re-seeded, 2nd backward: x.grad =", x.grad.numpy(), " (4x: summed)")
tape.zero()                                            # zero every gradient the tape knows
print("after tape.zero():       x.grad =", x.grad.numpy())

## 3. MJWarp functions inside a tape (4.0.1)
Every public function is wrapped by `diff._wrap`; no rule is registered, so every call inside an active tape raises `NotImplementedError`. Setup runs before the tape.

In [ ]:
# Example 29.2 - ex2_tape_error.py
"""Example 29.2 - in mujoco-warp 4.0.1 no public function can run inside an active wp.Tape.

mujoco_warp/__init__.py ends with _differentiable(locals()): every public function is wrapped by
diff._wrap. Outside a tape the wrapper calls straight through. Inside a tape it looks up
diff.RULES[name]; 4.0.1 registers no rule for any API function, so it raises before touching
the arguments. Setup calls (put_model, make_data, forward) therefore run before the tape.
Run: python ex2_tape_error.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import diff   # private API: the differentiability registry

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

# setup OUTSIDE any tape: these calls pass straight through the wrapper
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
mjd = mujoco.MjData(mjm)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2)
d.qpos.assign(np.tile(mjm.key_qpos[0], (2, 1)))     # keyframe "horizontal"
mjw.forward(m, d)
print("untaped forward ok, qacc[0] =", np.round(d.qacc.numpy()[0], 4))
print("mjw.step is a wrapper of", mjw.step.__wrapped__.__module__ + "." + mjw.step.__name__)

for name, call in [
    ("step", lambda: mjw.step(m, d)),
    ("forward", lambda: mjw.forward(m, d)),
    ("put_data", lambda: mjw.put_data(mjm, mjd, nworld=2)),
]:
    try:
        with wp.Tape() as tape:
            call()
        print(f"{name}: recorded (unexpected)")
    except NotImplementedError as e:
        print(f"{name:9s} -> NotImplementedError: {e}  | launches recorded: {len(tape.launches)}")

print("wrapped public functions:", len(diff.RULES))
print("registered rules:", sorted(k for k, v in diff.RULES.items() if v is not None))

## 4. `register_adjoint` on your own wrapped function
A custom backward must `+=` into input gradients and consume the output gradients.

In [ ]:
# Example 29.3 - ex3_register_adjoint.py
"""Example 29.3 - give your own wrapped function an adjoint rule with diff.register_adjoint.

square(m, d, x, y) writes y = x**2 with a kernel compiled WITHOUT a backward (like every MJWarp
kernel). diff._wrap makes it a "differentiable API function" exactly like mjw.step, and
register_adjoint attaches a custom backward: it must += into x.grad and then consume y.grad.
Under a wp.Tape the wrapper validates the call, runs the forward on a scratch tape and records
our backward; tape.backward() calls it.
Run: python ex3_register_adjoint.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import diff   # private API: register_adjoint, _wrap, RULES

np.set_printoptions(precision=4, suppress=True)


@wp.kernel(enable_backward=False)          # no generated adjoint, as in MJWarp's own kernels
def square_kernel(x: wp.array[float], y_out: wp.array[float]):
    i = wp.tid()
    y_out[i] = x[i] * x[i]


@wp.kernel(enable_backward=False)
def square_bwd_kernel(x: wp.array[float], y_grad: wp.array[float], x_grad_out: wp.array[float]):
    i = wp.tid()
    x_grad_out[i] += 2.0 * x[i] * y_grad[i]   # accumulate (+=), never overwrite


def square(m, d, x, y):
    """User op with the API calling convention (m, d, ...)."""
    wp.launch(square_kernel, dim=x.shape[0], inputs=[x], outputs=[y])


def square_bwd(m, d, x, y):
    """Called by tape.backward() with the forward arguments and the tape paused."""
    wp.launch(square_bwd_kernel, dim=x.shape[0], inputs=[x, y.grad], outputs=[x.grad])
    y.grad.zero_()   # consume: y was overwritten, so the cotangent of its old contents is 0


w = diff._wrap(square)                     # what _differentiable does to every mjw function
print("after _wrap:            RULES['square'] =", diff.RULES["square"])
diff.register_adjoint(square, square_bwd, array_args=("x", "y"))
rule = diff.RULES["square"]
print("after register_adjoint: backward =", rule.backward.__name__,
      "| array_args =", rule.array_args,
      "| in/out fields =", rule.in_fields, rule.out_fields)

mjm = mujoco.MjModel.from_xml_string("<mujoco><worldbody/></mujoco>")
m = mjw.put_model(mjm)
d = mjw.make_data(mjm)

x_np = np.array([1.5, -0.5, 2.0], dtype=np.float32)
x = wp.array(x_np, dtype=float)            # no requires_grad: array_args are promoted for us
y = wp.zeros(3, dtype=float)

w(m, d, x, y)                              # no tape: the wrapper calls square() directly
print("untaped: y =", y.numpy(), "| x.requires_grad =", x.requires_grad)

with wp.Tape() as tape:
    w(m, d, x, y)
print("taped:   recorded entries =", len(tape.launches), "| x.requires_grad =", x.requires_grad)
y.grad.fill_(1.0)
tape.backward()
print("x.grad =", x.grad.numpy(), " (2x; 3.0 for x = 1.5)")
print("y.grad =", y.grad.numpy(), " (consumed by the rule)")

eps = 1e-2                                 # float32 central difference through the same kernel
xp, xm = wp.array(x_np + eps, dtype=float), wp.array(x_np - eps, dtype=float)
yp, ym = wp.zeros(3, dtype=float), wp.zeros(3, dtype=float)
square(m, d, xp, yp)
square(m, d, xm, ym)
fd = (yp.numpy() - ym.numpy()) / (2 * eps)
print("FD     =", fd, "| max diff =", float(np.abs(fd - x.grad.numpy()).max()))

diff.RULES.pop("square", None)                 # undo: _wrap and register_adjoint edit global registries
diff._PARAMS.pop("square", None)

### Rules on Data fields, `supported()`, undeclared fields, native rules

In [ ]:
# Example 29.4 - ex4_adjoint_fields.py
"""Example 29.4 - rules on Data fields: in_fields, out_fields, a supported() hook, the
"not declared by this rule" guard, and why a native rule cannot work for an MJWarp function.

spring(m, d) writes d.qfrc_applied = -K * d.qpos (it reads one Data field and writes another).
Its rule declares in_fields=("qpos",) and out_fields=("qfrc_applied",) so the wrapper turns on
gradients for both, and a supported() hook refuses models with nv > 6. A native Warp loss kernel
recorded on the same tape composes with the custom rule: L = 0.5 * sum(qfrc_applied**2), so
dL/dqpos = K**2 * qpos.
Run: python ex4_adjoint_fields.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import diff   # private API

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


K = 3.0


@wp.kernel(enable_backward=False)
def spring_kernel(k: float, qpos_in: wp.array2d[float], qfrc_out: wp.array2d[float]):
    w, i = wp.tid()
    qfrc_out[w, i] = -k * qpos_in[w, i]          # hinge-only model: nq == nv


@wp.kernel(enable_backward=False)
def spring_bwd_kernel(k: float, qfrc_grad: wp.array2d[float], qpos_grad_out: wp.array2d[float]):
    w, i = wp.tid()
    qpos_grad_out[w, i] += -k * qfrc_grad[w, i]


@wp.kernel(enable_backward=False)
def spring_damper_kernel(k: float, qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                         qfrc_out: wp.array2d[float]):
    w, i = wp.tid()
    qfrc_out[w, i] = -k * qpos_in[w, i] - 0.5 * qvel_in[w, i]


@wp.kernel                                       # plain Warp kernel WITH an adjoint
def half_sq_loss(x: wp.array2d[float], loss: wp.array[float]):
    w, i = wp.tid()
    wp.atomic_add(loss, 0, 0.5 * x[w, i] * x[w, i])


def spring(m, d):
    wp.launch(spring_kernel, dim=(d.nworld, m.nv), inputs=[K, d.qpos], outputs=[d.qfrc_applied])


def spring_bwd(m, d):
    wp.launch(spring_bwd_kernel, dim=(d.nworld, m.nv), inputs=[K, d.qfrc_applied.grad],
              outputs=[d.qpos.grad])
    d.qfrc_applied.grad.zero_()                  # overwritten output: consumed


def spring_ok(m, d):
    return f"nv = {m.nv} > 6" if m.nv > 6 else None   # None / () / [] mean "supported"


def spring_damper(m, d):
    wp.launch(spring_damper_kernel, dim=(d.nworld, m.nv), inputs=[K, d.qpos, d.qvel],
              outputs=[d.qfrc_applied])


w_spring = diff._wrap(spring)
diff.register_adjoint(spring, spring_bwd, in_fields=("qpos",), out_fields=("qfrc_applied",),
                      supported=spring_ok)

# 1. the gradient on a Data field, composed with a native loss kernel on the same tape
NWORLD = 3
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
d.qpos.assign(np.array([[0.1, -0.2], [0.5, 0.0], [1.0, 0.3]]))
loss = wp.zeros(1, dtype=float, requires_grad=True)
with wp.Tape() as tape:
    w_spring(m, d)                               # custom rule (recorded as a function)
    wp.launch(half_sq_loss, dim=(NWORLD, m.nv), inputs=[d.qfrc_applied], outputs=[loss])
print("recorded:", len(tape.launches), "entries | loss =", round(float(loss.numpy()[0]), 4))
loss.grad.fill_(1.0)
tape.backward()
print("dL/dqpos (tape)   =\n", d.qpos.grad.numpy())
print("K^2 * qpos (exact) =\n", K * K * d.qpos.numpy())

# 2. supported() refuses a model with nv > 6: raised before anything is recorded
mjm5 = mujoco.MjModel.from_xml_string(box_stack(2))
m5, d5 = mjw.put_model(mjm5), mjw.make_data(mjm5, nconmax=8, njmax=32)
try:
    with wp.Tape() as tape5:
        w_spring(m5, d5)
except NotImplementedError as e:
    print("NotImplementedError:", e, "| recorded:", len(tape5.launches))

# 3. a rule that forgets a field the call reads. qvel already has gradients (an earlier rule,
#    or the user, enabled them), so silently reading it would drop its gradient: the wrapper
#    checks the scratch tape's launches and raises.
w_sd = diff._wrap(spring_damper)
diff.register_adjoint(spring_damper, lambda m, d: None,
                      in_fields=("qpos",), out_fields=("qfrc_applied",))
d.qvel.requires_grad = True
try:
    with wp.Tape():
        w_sd(m, d)
except NotImplementedError as e:
    print("NotImplementedError:", e)

# 4. a "native" rule (no backward) asks Warp to replay the forward's own kernel adjoints. MJWarp
#    kernels are compiled with enable_backward=False, so for an API function this must raise.
diff.register_adjoint("kinematics", in_fields=("qpos",), out_fields=("xpos",))
try:
    with wp.Tape():
        mjw.kinematics(m, d)
except NotImplementedError as e:
    print("NotImplementedError:", e)
diff.RULES["kinematics"] = None                  # undo: back to the shipped 4.0.1 state
for n in ("spring", "spring_damper"):          # and drop the rules this script added
    diff.RULES.pop(n, None)
    diff._PARAMS.pop(n, None)

## 5. Finite differences across worlds
One world per perturbation, all in one batched rollout. Compare epsilons in float32 against a float64 C MuJoCo reference: too big gives truncation error, too small gives round-off.

In [ ]:
# Example 29.5 - ex5_fd_jacobian.py
"""Example 29.5 - a Jacobian by central finite differences, every perturbation its own world.

CARTPOLE starts hanging (hinge = pi) and gets a piecewise-constant force: KNOTS control values,
each held for HOLD steps. We want J = d(final cart x, final pole angle) / d(knot values), a
2 x KNOTS matrix. World 0 is the nominal; for each epsilon and each knot j, one world adds +eps
to knot j and one adds -eps. All of them run in ONE batched rollout (2 * KNOTS * len(EPS) + 1
worlds). The float64 C MuJoCo Jacobian is the reference, so we can see how the float32 error
depends on epsilon.
Run: python ex5_fd_jacobian.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

KNOTS, HOLD = 5, 10                        # 5 control values x 10 steps = 0.5 s
U0 = np.array([0.3, -0.2, 0.4, 0.1, -0.3])  # nominal knot values (inside ctrlrange)
EPS = [1e-1, 3e-2, 1e-2, 3e-3, 1e-3, 3e-4, 1e-4, 3e-5, 1e-5]
NWORLD = 1 + 2 * KNOTS * len(EPS)          # 91: the same on CPU and GPU (tiny batch)


@wp.kernel
def set_knot(sched: wp.array2d[float], k: int, ctrl_out: wp.array2d[float]):
    w = wp.tid()
    ctrl_out[w, 0] = sched[w, k]


# one row of knot values per world: nominal, then (+eps_j, -eps_j) blocks per epsilon
sched = np.tile(U0, (NWORLD, 1))
for e, eps in enumerate(EPS):
    for j in range(KNOTS):
        sched[1 + 2 * (e * KNOTS + j), j] += eps
        sched[2 + 2 * (e * KNOTS + j), j] -= eps

mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
q0 = mjm.qpos0.copy()
q0[1] = np.pi                              # pole hanging down
d.qpos.assign(np.tile(q0, (NWORLD, 1)))
sched_wp = wp.array(sched, dtype=float)
for k in range(KNOTS):                     # one batched rollout for all perturbations
    wp.launch(set_knot, dim=NWORLD, inputs=[sched_wp, k], outputs=[d.ctrl])
    for _ in range(HOLD):
        mjw.step(m, d)
out = d.qpos.numpy().astype(np.float64)    # (NWORLD, 2): final cart x, pole angle


def c_rollout(u):
    """float64 C MuJoCo, same schedule: the reference."""
    dd = mujoco.MjData(mjm)
    dd.qpos[:] = q0
    for k in range(KNOTS):
        dd.ctrl[0] = u[k]
        for _ in range(HOLD):
            mujoco.mj_step(mjm, dd)
    return dd.qpos.copy()


J_ref = np.zeros((2, KNOTS))
for j in range(KNOTS):
    up, um = U0.copy(), U0.copy()
    up[j] += 1e-6
    um[j] -= 1e-6
    J_ref[:, j] = (c_rollout(up) - c_rollout(um)) / 2e-6

print(f"nworld = {NWORLD} (1 nominal + 2 x {KNOTS} knots x {len(EPS)} epsilons), "
      f"{KNOTS * HOLD} steps")
print("nominal final [x, theta]: MJWarp", out[0], "| C MuJoCo", c_rollout(U0))
errs = []
for e, eps in enumerate(EPS):
    J = np.zeros((2, KNOTS))
    for j in range(KNOTS):
        r = 1 + 2 * (e * KNOTS + j)
        J[:, j] = (out[r] - out[r + 1]) / (2 * eps)
    err = float(np.abs(J - J_ref).max() / np.abs(J_ref).max())
    errs.append(err)
    print(f"eps = {eps:.0e}: max rel. error vs float64 = {err:.2e}")
    if eps == 1e-2:
        J_show = J
print("J (eps = 1e-2), rows = [x, theta], cols = knots:\n", J_show)
print("J reference (C MuJoCo, float64):\n", J_ref)
print("float32 machine epsilon:", np.finfo(np.float32).eps)

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"eps": EPS, "err": [round(v, 6) for v in errs],
                   "J": np.round(J_show, 4).tolist(), "J_ref": np.round(J_ref, 4).tolist()},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. CEM: iterative sampling-based optimization
Snapshot with `State.PHYSICS | TIME` once, restore it every iteration, sample NWORLD plans on the device, refit a Gaussian to the elites.

In [ ]:
# Example 29.6 - ex6_cem.py
"""Example 29.6 - the cross-entropy method (CEM): Ch 8's random shooting, made iterative.

CARTPOLE starts hanging (hinge = pi). The plan is an open-loop sequence of KNOTS force values,
each held for HOLD steps. CEM keeps a Gaussian over plans (mean mu, std sigma per knot). Every
iteration: restore the start snapshot (State.PHYSICS | TIME) in all worlds, sample NWORLD plans
on the device, roll them all out at once while a kernel sums the reward per world, keep the
NELITE best plans, refit mu and sigma to them, repeat. World 0 always replays the current mean
and world 1 the best plan found so far (elitism), so the best return never goes down.
Run: python ex6_cem.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 64     # samples per CEM iteration
NELITE = NWORLD // 8                # elites kept to refit the Gaussian
NITER = 12 if ON_GPU else 10
KNOTS, HOLD = 10, 20                # 10 knots x 20 steps = 2 s plan
SIGMA0, SIGMA_MIN = 0.8, 0.05


@wp.kernel
def sample_plans(seed: int, it: int, mu: wp.array[float], sigma: wp.array[float],
                 best: wp.array[float], plans_out: wp.array2d[float]):
    w, k = wp.tid()
    rng = wp.rand_init(seed, (it * 65536 + w) * 64 + k)
    u = mu[k]                                  # world 0 replays the mean plan
    if w == 1:
        u = best[k]                            # world 1 replays the best plan so far
    elif w > 1:
        u = mu[k] + sigma[k] * wp.randn(rng)
    plans_out[w, k] = wp.clamp(u, -1.0, 1.0)   # sample, then clip to ctrlrange


@wp.kernel
def set_ctrl(plans_in: wp.array2d[float], k: int, ctrl_out: wp.array2d[float]):
    w = wp.tid()
    ctrl_out[w, 0] = plans_in[w, k]


@wp.kernel
def add_reward(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
               ctrl_in: wp.array2d[float], ret_out: wp.array[float]):
    w = wp.tid()
    x, theta = qpos_in[w, 0], qpos_in[w, 1]    # theta = 0 is upright
    u, om = ctrl_in[w, 0], qvel_in[w, 1]
    ret_out[w] += wp.cos(theta) - 0.1 * x * x - 0.01 * u * u - 0.001 * om * om


mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
q0 = mjm.qpos0.copy()
q0[1] = np.pi                                  # hanging start
d.qpos.assign(np.tile(q0, (NWORLD, 1)))

SIG = int(mjw.State.PHYSICS | mjw.State.TIME)  # Ch 8 sec. 4; no contacts, so no WARMSTART
snap = wp.zeros((NWORLD, mujoco.mj_stateSize(mjm, SIG)), dtype=float)
mjw.get_state(m, d, snap, SIG)                 # the start state, taken once

mu = np.zeros(KNOTS)
sigma = np.full(KNOTS, SIGMA0)
plans = wp.zeros((NWORLD, KNOTS), dtype=float)
ret = wp.zeros(NWORLD, dtype=float)
mu_wp, sigma_wp = wp.zeros(KNOTS, dtype=float), wp.zeros(KNOTS, dtype=float)
best_wp = wp.zeros(KNOTS, dtype=float)
hist, replay_ok, prev_best = [], [], None
print(f"nworld {NWORLD} samples/iter, {NELITE} elites, plan {KNOTS} knots x {HOLD} steps")
for it in range(NITER):
    mjw.set_state(m, d, snap, SIG)             # every world back to the start
    mu_wp.assign(mu)
    sigma_wp.assign(sigma)
    wp.launch(sample_plans, dim=(NWORLD, KNOTS), inputs=[11, it, mu_wp, sigma_wp, best_wp],
              outputs=[plans])
    ret.zero_()
    for k in range(KNOTS):                     # all NWORLD rollouts in one batch
        wp.launch(set_ctrl, dim=NWORLD, inputs=[plans, k], outputs=[d.ctrl])
        for _ in range(HOLD):
            mjw.step(m, d)
            wp.launch(add_reward, dim=NWORLD, inputs=[d.qpos, d.qvel, d.ctrl], outputs=[ret])
    r = ret.numpy()                            # one small host copy per iteration
    if prev_best is not None:
        replay_ok.append(bool(r[1] == prev_best))  # same plan + same snapshot = same return
    P = plans.numpy()
    elite = np.argsort(r)[-NELITE:]            # top-NELITE worlds
    mu = P[elite].mean(axis=0)
    sigma = np.maximum(P[elite].std(axis=0), SIGMA_MIN)
    best = int(elite[-1])
    best_wp.assign(P[best])                    # elitism: carried into the next iteration
    prev_best = r[best]
    hist.append({"best": round(float(r[best]), 2), "mean_plan": round(float(r[0]), 2),
                 "elite_min": round(float(r[elite[0]]), 2), "sigma": round(float(sigma.mean()), 3),
                 "mu": np.round(mu, 3).tolist(),
                 "samples": np.round(P[2:26, :], 3).tolist(),
                 "elite_idx": [int(e) - 2 for e in elite if 1 < e < 26]})
    print(f"iter {it}: best return {r[best]:7.2f} | mean-plan return {r[0]:7.2f} | "
          f"mean sigma {sigma.mean():.3f}")

# replay the best plan in world 1 and report how close to upright the pole gets
mjw.set_state(m, d, snap, SIG)
wp.launch(sample_plans, dim=(NWORLD, KNOTS), inputs=[11, 0, mu_wp, sigma_wp, best_wp],
          outputs=[plans])
theta, frames = [], []
for k in range(KNOTS):
    wp.launch(set_ctrl, dim=NWORLD, inputs=[plans, k], outputs=[d.ctrl])
    for s in range(HOLD):
        mjw.step(m, d)
        if args.dump_json and s % 4 == 3:      # replay frames for the page (dump only)
            frames.append(np.round(d.qpos.numpy()[1], 3).tolist())
    theta.append(float(d.qpos.numpy()[1, 1]))
th = np.array(theta)
print("world-1 replays of the carried best plan reproduced its return exactly:", all(replay_ok))
print("best plan:", best_wp.numpy())
print("pole angle every 0.2 s (0 = upright):", th)
print(f"min |angle - 0| during the plan: {np.abs(np.angle(np.exp(1j * th))).min():.3f} rad")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "nelite": NELITE, "knots": KNOTS, "hold": HOLD,
                   "dt": float(mjm.opt.timestep), "iters": hist,
                   "theta_final": np.round(th, 3).tolist(),
                   "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 7. Capstone: parallel system identification
One world per candidate (damping, mass); batched Model fields plus `mjw.set_const` after each mass write, eagerly and outside any graph; loss summed on the device; full-covariance CEM.

In [ ]:
# Example 29.7 - ex7_sysid_capstone.py
"""Example 29.7 - capstone: parallel system identification of a double pendulum.

A "real" PENDULUM (C MuJoCo, float64, plus sensor noise) has a hidden joint damping and a hidden
mass of the lower link. We record its joint angles while a known torque drives the shoulder.
Then every MJWarp world gets one candidate (damping, mass): batched Model fields (Ch 10), with
mjw.set_const called eagerly after each mass write (outside any graph) so the derived constants
match. All candidates replay the same torque in one batched rollout, a kernel sums the squared
angle error per world on the device, and 5 CEM rounds shrink the search onto the truth. The
damping is sharply determined by the recording, the mass only weakly, with a mild positive coupling
between them. This CEM refits a full 2x2 covariance to the elites: cheap insurance against a tilted
valley, not needed here (one sigma per parameter recovers the same values on this problem).
Run: python ex7_sysid_capstone.py [--dump-json PATH]
  --dump-json PATH also evaluates the loss on a grid of candidates (the landscape figure).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 64        # candidates per CEM round
NELITE = max(4, NWORLD // 8)
NROUND = 5
T = 150                                # 1.5 s of data at dt = 0.01
TRUE_DAMPING, TRUE_MASS = 0.15, 1.6    # hidden from the identification code
LO, HI = np.array([0.0, 0.5]), np.array([0.5, 3.0])   # prior box: damping [N m s], mass [kg]
TRUTH_NOISE = 0.002                    # rad, sensor noise on the recording

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
LINK2 = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "link2")
torque = 2.0 * np.sin(2 * np.pi * 1.0 * np.arange(T) * mjm.opt.timestep)   # known excitation

# ---- 1. the "real" system: float64 C MuJoCo with the hidden parameters, noisy sensors ----
real = mujoco.MjModel.from_xml_string(PENDULUM)
real.dof_damping[:] = TRUE_DAMPING
real.body_inertia[LINK2] *= TRUE_MASS / real.body_mass[LINK2]
real.body_mass[LINK2] = TRUE_MASS
mujoco.mj_setConst(real, mujoco.MjData(real))
rd = mujoco.MjData(real)
mujoco.mj_resetDataKeyframe(real, rd, 0)
obs = np.zeros((T, mjm.nq))
for t in range(T):
    rd.ctrl[0] = torque[t]
    mujoco.mj_step(real, rd)
    obs[t] = rd.qpos
obs += np.random.default_rng(0).normal(0.0, TRUTH_NOISE, obs.shape)


# ---- 2. the batched candidate simulator ----
@wp.kernel
def set_torque(tau: wp.array[float], t: int, ctrl_out: wp.array2d[float]):
    w = wp.tid()
    ctrl_out[w, 0] = tau[t]


@wp.kernel
def add_loss(qpos_in: wp.array2d[float], obs_in: wp.array2d[float], t: int, nq: int,
             loss_out: wp.array[float]):
    w = wp.tid()
    acc = float(0.0)
    for i in range(nq):
        e = qpos_in[w, i] - obs_in[t, i]
        acc += e * e
    loss_out[w] += acc


FIELDS = ["dof_damping", "body_mass", "body_inertia",                       # what we vary
          "body_subtreemass", "body_invweight0", "dof_invweight0", "actuator_acc0"]  # derived


def make_candidates(n):
    m = mjw.put_model(mjm, batch_sizes={k: n for k in FIELDS})
    d = mjw.make_data(mjm, nworld=n)
    return m, d


def evaluate(m, d, params, tau_wp, obs_wp, loss):
    """Mean squared angle error for every row of params = [[damping, mass], ...]."""
    n = params.shape[0]
    m.dof_damping.assign(np.repeat(params[:, :1], mjm.nv, axis=1))
    bm, bi = np.tile(mjm.body_mass, (n, 1)), np.tile(mjm.body_inertia, (n, 1, 1))
    bm[:, LINK2] = params[:, 1]
    bi[:, LINK2] *= (params[:, 1] / mjm.body_mass[LINK2])[:, None]
    m.body_mass.assign(bm)
    m.body_inertia.assign(bi)
    mjw.set_const(m, d)                # derived constants, per world; eager, outside any graph
    mjw.reset_data_keyframe(m, d, 0)   # every world starts at the "horizontal" keyframe
    loss.zero_()
    for t in range(T):
        wp.launch(set_torque, dim=n, inputs=[tau_wp, t], outputs=[d.ctrl])
        mjw.step(m, d)
        wp.launch(add_loss, dim=n, inputs=[d.qpos, obs_wp, t, mjm.nq], outputs=[loss])
    return loss.numpy() / (T * mjm.nq)


tau_wp = wp.array(torque, dtype=float)
obs_wp = wp.array(obs, dtype=float)
m, d = make_candidates(NWORLD)
loss = wp.zeros(NWORLD, dtype=float)
print("batched shapes: dof_damping", m.dof_damping.shape, "| body_mass", m.body_mass.shape,
      "| body_subtreemass", m.body_subtreemass.shape)

# ---- 3. CEM over (damping, mass) ----
rng = np.random.default_rng(1)
mu, cov = (LO + HI) / 2, np.diag(((HI - LO) / 2) ** 2)
rounds = []
for r in range(NROUND):
    P = np.clip(rng.multivariate_normal(mu, cov, size=NWORLD), LO, HI)
    P[0] = mu                                         # world 0 evaluates the current mean
    L = evaluate(m, d, P, tau_wp, obs_wp, loss)
    elite = np.argsort(L)[:NELITE]
    mu = P[elite].mean(axis=0)
    cov = np.cov(P[elite].T) + 1e-8 * np.eye(2)       # full covariance: allows a tilted Gaussian
    sd, corr = np.sqrt(np.diag(cov)), cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    rounds.append({"samples": np.round(P[:64], 4).tolist(), "elite": [int(e) for e in elite
                   if e < 64], "mu": np.round(mu, 4).tolist(), "cov": np.round(cov, 7).tolist()})
    print(f"round {r}: best loss {L[elite[0]]:.2e} | damping {mu[0]:.4f} +- {sd[0]:.4f}"
          f" | mass {mu[1]:.4f} +- {sd[1]:.4f} | corr {corr:+.2f}")

print(f"recovered: damping {mu[0]:.4f} (true {TRUE_DAMPING}), mass {mu[1]:.4f} (true {TRUE_MASS})")
L2 = evaluate(m, d, np.tile([mu, [TRUE_DAMPING, TRUE_MASS]], (NWORLD // 2, 1)), tau_wp, obs_wp,
              loss)
print(f"rms angle error: fit {np.sqrt(L2[0]):.4f} rad, truth {np.sqrt(L2[1]):.4f} rad "
      f"(sensor noise {TRUTH_NOISE} rad)")

if args.dump_json:
    G = 32 if ON_GPU else 16                          # landscape: G x G candidates, one batch
    gd, gm = np.linspace(LO[0], HI[0], G), np.linspace(LO[1], HI[1], G)
    grid = np.array([[a, b] for b in gm for a in gd])
    mg, dg = make_candidates(G * G)
    Lg = evaluate(mg, dg, grid, tau_wp, obs_wp, wp.zeros(G * G, dtype=float))
    print(f"landscape: {G}x{G} grid, min loss at damping {grid[Lg.argmin(), 0]:.3f},"
          f" mass {grid[Lg.argmin(), 1]:.3f}")
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "T": T, "lo": LO.tolist(), "hi": HI.tolist(),
                   "truth": [TRUE_DAMPING, TRUE_MASS], "grid_n": G,
                   "loss_grid": np.round(np.log10(Lg), 3).reshape(G, G).tolist(),
                   "rounds": rounds, "fit": np.round(mu, 4).tolist()}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

### With contacts: identify sliding friction (no `set_const` needed)

In [ ]:
# Example 29.8 - ex8_sysid_friction.py
"""Example 29.8 - identify sliding friction from a box sliding down a slope (contacts).

The "real" box (float64 C MuJoCo) slides down a 30 degree slope with a hidden friction
coefficient; we record its distance along the slope (1 mm sensor noise). Every MJWarp world
gets one candidate mu in a batched geom_friction (Ch 10 sec. 3): a friction change needs no
set_const. One batched rollout scores all candidates; each round keeps the best and shrinks the
search interval around it (grid refinement). World 0 re-scores the best value so far.
Run: python ex8_sysid_friction.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

SLOPE_DEG = 30.0
n = np.array([np.sin(np.radians(SLOPE_DEG)), 0.0, np.cos(np.radians(SLOPE_DEG))])  # plane normal
p0 = 0.101 * n                                    # box centre: half-size + 1 mm above the plane
SLOPE = f"""
<mujoco model="slope">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="slope" type="plane" size="5 5 .1" euler="0 {SLOPE_DEG} 0"/>
    <body name="box" pos="{p0[0]:.5f} 0 {p0[2]:.5f}" euler="0 {SLOPE_DEG} 0">
      <freejoint/>
      <geom name="box" type="box" size=".1 .1 .1" mass="1" rgba=".3 .3 1 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 16                   # candidates per round
NROUND = 4
T = 200                                           # 1 s at dt = 0.005
TRUE_MU = 0.35                                    # hidden
DOWN = np.array([np.cos(np.radians(SLOPE_DEG)), 0.0, -np.sin(np.radians(SLOPE_DEG))])  # downhill

mjm = mujoco.MjModel.from_xml_string(SLOPE)

# ---- 1. the "real" recording: C MuJoCo, float64, sensor noise ----
real = mujoco.MjModel.from_xml_string(SLOPE)
real.geom_friction[:, 0] = TRUE_MU
rd = mujoco.MjData(real)
obs = np.zeros(T)
for t in range(T):
    mujoco.mj_step(real, rd)
    obs[t] = (rd.qpos[:3] - mjm.qpos0[:3]) @ DOWN  # distance travelled downhill
obs += np.random.default_rng(0).normal(0.0, 0.001, T)
print(f"real box: {obs[-1]:.3f} m downhill after {T * mjm.opt.timestep:.1f} s")


@wp.kernel
def add_loss(qpos_in: wp.array2d[float], p0: wp.vec3, down: wp.vec3, obs_in: wp.array[float],
             t: int, loss_out: wp.array[float]):
    w = wp.tid()
    p = wp.vec3(qpos_in[w, 0], qpos_in[w, 1], qpos_in[w, 2])
    e = wp.dot(p - p0, down) - obs_in[t]
    loss_out[w] += e * e


m = mjw.put_model(mjm, batch_sizes={"geom_friction": NWORLD})
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
obs_wp = wp.array(obs, dtype=float)
loss = wp.zeros(NWORLD, dtype=float)
P0, DN = wp.vec3(*mjm.qpos0[:3]), wp.vec3(*DOWN)


def evaluate(mu):
    fr = m.geom_friction.numpy()
    fr[:, :, 0] = mu[:, None]                     # slope AND box (contacts take the max)
    m.geom_friction.assign(fr)
    mjw.reset_data(m, d)                          # qpos0, zero qvel, zero warmstart
    loss.zero_()
    for t in range(T):
        mjw.step(m, d)
        wp.launch(add_loss, dim=NWORLD, inputs=[d.qpos, P0, DN, obs_wp, t], outputs=[loss])
    return np.sqrt(loss.numpy() / T)              # rms distance error [m]


lo, hi, best = 0.05, 0.55, 0.3                   # prior interval, any starting guess
curves = []
for r in range(NROUND):
    mu = np.concatenate([[best], np.linspace(lo, hi, NWORLD - 1)])
    err = evaluate(mu)
    b = int(err.argmin())
    best, step = mu[b], (hi - lo) / (NWORLD - 2)
    curves.append({"mu": np.round(mu[1:], 5).tolist(), "err": np.round(err[1:], 5).tolist()})
    print(f"round {r}: mu in [{lo:.4f}, {hi:.4f}] step {step:.4f} -> best mu "
          f"{best:.4f}, rms error {err[b] * 1000:.2f} mm")
    lo, hi = max(best - 2 * step, 0.0), best + 2 * step   # zoom to +- two grid steps
print(f"recovered mu = {best:.4f} (true {TRUE_MU})")
print(f"Coulomb check: tan(slope) = {np.tan(np.radians(SLOPE_DEG)):.3f} > mu, so the box slides")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "T": T, "truth": TRUE_MU, "rounds": curves,
                   "fit": round(float(best), 5)}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## Exercise 29.1 (auto-checked)
FD gradient of a ball's landing time with respect to its drop height, checked against the analytic value.

In [ ]:
# Exercise 29.1 - FD gradient of the landing time w.r.t. the drop height (auto-checked)
# Drop BALL_DROP from H - EPS and H + EPS in TWO worlds of one batch, find the time at which the
# ball centre first reaches z = R (interpolate between steps), and form the central difference.
# Analytic: t = sqrt(2 (H - R) / g), so dt/dH = 1 / sqrt(2 g (H - R)).
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

H, EPS, R, G = 1.0, 0.05, 0.1, 9.81
dtdh = ...       # your FD estimate
analytic = 1.0 / np.sqrt(2 * G * (H - R))
if dtdh is ...:
  print("Exercise 29.1: not attempted")
else:
  assert abs(dtdh - analytic) / analytic < 0.03
  print("Exercise 29.1: ok")

In [ ]:
#@title Exercise 29.1: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""
H, EPS, R, G = 1.0, 0.05, 0.1, 9.81


def landing_times(heights):
    """Time at which the ball centre first reaches z = R, linearly interpolated between steps."""
    mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=len(heights), nconmax=4, njmax=16)
    q = np.tile(mjm.qpos0, (len(heights), 1))
    q[:, 2] = heights
    d.qpos.assign(q)
    dt = mjm.opt.timestep
    t_land = np.full(len(heights), np.nan)
    z_prev = np.array(heights, dtype=float)
    for i in range(1, 400):
        mjw.step(m, d)
        z = d.qpos.numpy()[:, 2].copy()   # copy: on CPU .numpy() is a view
        hit = np.isnan(t_land) & (z <= R)
        t_land[hit] = (i - 1) * dt + dt * (z_prev[hit] - R) / (z_prev[hit] - z[hit])
        z_prev = z
        if not np.isnan(t_land).any():
            break
    return t_land


t = landing_times([H - EPS, H + EPS])
dtdh = (t[1] - t[0]) / (2 * EPS)
analytic = 1.0 / np.sqrt(2 * G * (H - R))
print(f"FD dt/dh = {dtdh:.4f} s/m, analytic = {analytic:.4f} s/m")
assert abs(dtdh - analytic) / analytic < 0.03
print("ok")

## Exercise 29.2 (auto-checked)
A custom adjoint rule for `y = 2x + 1`, checked with `wp.Tape`.

In [ ]:
# Exercise 29.2 - a custom adjoint for y = 2x + 1 (auto-checked)
# Write affine(m, d, x, y) and affine_bwd(m, d, x, y), wrap with diff._wrap, register with
# diff.register_adjoint(..., array_args=("x", "y")), run it under wp.Tape, seed y.grad = 1.
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import diff

y = ...  # ... your code: define x and y, run under wp.Tape, call tape.backward() ...
if y is ...:
  print("Exercise 29.2: not attempted")
else:
  assert np.allclose(y.numpy(), [1.0, 3.0, -5.0])   # x = [0, 1, -3]
  assert np.allclose(x.grad.numpy(), 2.0)
  assert np.allclose(y.grad.numpy(), 0.0)           # the rule consumed the output cotangent
  print("Exercise 29.2: ok")

In [ ]:
#@title Exercise 29.2: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import diff


@wp.kernel(enable_backward=False)
def affine_kernel(x: wp.array[float], y_out: wp.array[float]):
    i = wp.tid()
    y_out[i] = 2.0 * x[i] + 1.0


@wp.kernel(enable_backward=False)
def affine_bwd_kernel(y_grad: wp.array[float], x_grad_out: wp.array[float]):
    i = wp.tid()
    x_grad_out[i] += 2.0 * y_grad[i]


def affine(m, d, x, y):
    wp.launch(affine_kernel, dim=x.shape[0], inputs=[x], outputs=[y])


def affine_bwd(m, d, x, y):
    wp.launch(affine_bwd_kernel, dim=x.shape[0], inputs=[y.grad], outputs=[x.grad])
    y.grad.zero_()


w = diff._wrap(affine)
diff.register_adjoint(affine, affine_bwd, array_args=("x", "y"))
mjm = mujoco.MjModel.from_xml_string("<mujoco><worldbody/></mujoco>")
m, d = mjw.put_model(mjm), mjw.make_data(mjm)
x = wp.array([0.0, 1.0, -3.0], dtype=float)
y = wp.zeros(3, dtype=float)
with wp.Tape() as tape:
    w(m, d, x, y)
y.grad.fill_(1.0)
tape.backward()
print("y =", y.numpy(), "x.grad =", x.grad.numpy())
assert np.allclose(y.numpy(), [1.0, 3.0, -5.0])
assert np.allclose(x.grad.numpy(), 2.0)
assert np.allclose(y.grad.numpy(), 0.0)
print("ok")
diff.RULES.pop("affine", None)    # leave the global registry as 4.0.1 ships it
diff._PARAMS.pop("affine", None)

## Stretch: friction and restitution together
Record a box thrown onto a floor in C MuJoCo with hidden sliding friction and a hidden bounce (use the direct form of `solref`, both numbers negative: `"-stiffness -damping"`, e.g. `solref="-5000 -2"` for stiffness 5000 and a low damping of 2; restitution needs per-world `geom_solref`). Identify both with a 2-D CEM over one batch per round, as in Example 29.7. Which parameter is better determined, and why? Plot the 2-D loss landscape.